# BoostFL-IoT on UNSW-NB15 (IID) — Flower

## 1. Imports and configuration

In [1]:
import copy
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

import flwr as fl

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [2]:
NUM_CLIENTS = 10
NUM_PARTITIONS = 10
NUM_ROUNDS = 15
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-3
SERVER_LR = 1.0

EPSILON = 1e-8
MIN_ALPHA = 0.1
MAX_ALPHA = 10.0

GPU_PER_CLIENT = 0.5 if torch.cuda.is_available() else 0.0
CPUS_PER_CLIENT = 2

## 2. Data loading (UNSW-NB15)

In [3]:
DATA_PATH = "../../data/UNSW-NB15.csv"


def load_unsw(file_path, target_column, test_size=0.2, random_state=SEED):
    df = pd.read_csv(file_path, low_memory=False).drop_duplicates()

    df[target_column] = df[target_column].astype(str).str.strip()
    df[target_column] = df[target_column].replace(["", "nan", "NaN"], "Normal")
    attack_mapping = {
        "Backdoors": "Backdoor",
        "Reconnaissance": "Reconnaissance",
        "Shellcolde": "Shellcode",
    }
    df[target_column] = df[target_column].replace(attack_mapping)
    df = df.dropna(subset=[target_column])

    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state,
        stratify=df[target_column],
    )

    num_cols = train_df.select_dtypes(include=[np.number]).columns
    cat_cols = train_df.select_dtypes(exclude=[np.number]).columns
    for col in num_cols:
        if train_df[col].isnull().any():
            m = train_df[col].median()
            train_df[col] = train_df[col].fillna(m)
            test_df[col] = test_df[col].fillna(m)
    for col in cat_cols:
        if train_df[col].isnull().any():
            mode = train_df[col].mode()
            fill = mode[0] if not mode.empty else "Unknown"
            train_df[col] = train_df[col].fillna(fill)
            test_df[col] = test_df[col].fillna(fill)

    y_train = train_df[target_column].astype(str).str.strip()
    y_test = test_df[target_column].astype(str).str.strip()

    cols_to_drop = [target_column, "subcategory", "attack_cat", "label",
                    "pkSeqID", "saddr", "daddr", "id"]
    X_train = train_df.drop(columns=cols_to_drop, errors="ignore").copy()
    X_test = test_df.drop(columns=cols_to_drop, errors="ignore").copy()

    non_numeric = list(
        set(X_train.select_dtypes(exclude=[np.number]).columns)
        | set(X_test.select_dtypes(exclude=[np.number]).columns)
    )
    for col in non_numeric:
        le = LabelEncoder()
        le.fit(X_train[col].astype(str))
        mapping = {c: i for i, c in enumerate(le.classes_)}
        X_train[col] = X_train[col].astype(str).map(mapping).fillna(-1).astype(int)
        X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1).astype(int)

    def safe_numeric(d):
        d = d.apply(lambda c: c.map(lambda v: str(v).strip() if isinstance(v, str) else v))
        d = d.apply(pd.to_numeric, errors="coerce")
        return d.replace([np.inf, -np.inf], np.nan).fillna(0)

    X_train = safe_numeric(X_train)
    X_test = safe_numeric(X_test)

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train.values)
    y_test_enc = label_encoder.transform(y_test.values)
    class_names = list(label_encoder.classes_)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train.values.astype(np.float64))
    X_test_s = scaler.transform(X_test.values.astype(np.float64))

    train_ds = TensorDataset(torch.from_numpy(X_train_s).float(),
                             torch.from_numpy(y_train_enc).long())
    test_ds = TensorDataset(torch.from_numpy(X_test_s).float(),
                            torch.from_numpy(y_test_enc).long())
    return train_ds, test_ds, class_names, len(class_names), X_train_s.shape[1]

In [4]:
def to_binary(train_ds, test_ds, class_names, normal_name="Normal"):
    """Collapse the multiclass task to Normal vs Attack (0 = Normal, 1 = Attack)."""
    names = [str(c).strip() for c in class_names]
    normal_id = names.index(normal_name) if normal_name in names else 0

    def convert(ds):
        X = ds.tensors[0]
        y = ds.tensors[1]
        y_bin = (y != normal_id).long()
        return TensorDataset(X, y_bin)

    return convert(train_ds), convert(test_ds), ["Normal", "Attack"], 2

## 3. IID partitioning

In [5]:
def partition_iid(dataset, num_partitions, num_classes, seed=SEED):
    rng = np.random.RandomState(seed)
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    idx_by_class = [np.where(labels == c)[0].tolist() for c in range(num_classes)]
    partitions = [[] for _ in range(num_partitions)]
    for c in range(num_classes):
        idx = idx_by_class[c]
        rng.shuffle(idx)
        per = len(idx) // num_partitions
        rem = len(idx) % num_partitions
        start = 0
        for p in range(num_partitions):
            extra = 1 if p < rem else 0
            end = start + per + extra
            partitions[p].extend(idx[start:end])
            start = end
    for p in range(num_partitions):
        rng.shuffle(partitions[p])
    return partitions

## 4. Weak learner and residual loss

In [6]:
class WeakLearner(nn.Module):
    """Shallow MLP weak learner."""
    def __init__(self, input_dim, num_classes):
        super(WeakLearner, self).__init__()
        self.fc1 = nn.Linear(input_dim, 200)
        self.fc2 = nn.Linear(200, 100)
        self.fc3 = nn.Linear(100, 50)
        self.fc4 = nn.Linear(50, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x


class ResidualLoss(nn.Module):
    """Smooth-L1 loss between weak-learner output and ensemble residual."""
    def forward(self, predictions, residuals):
        return F.smooth_l1_loss(predictions, residuals)


def get_ndarrays(net):
    return [v.detach().cpu().numpy() for _, v in net.state_dict().items()]


def set_ndarrays(net, params):
    sd = net.state_dict()
    net.load_state_dict({k: torch.tensor(v, device=DEVICE)
                         for k, v in zip(sd.keys(), params)}, strict=True)

## 5. Global boosting ensemble

In [7]:
class Ensemble:
    """Global boosting ensemble F_t = f0 + sum_i (alpha_i / sum alpha) h_i."""
    def __init__(self, f0, input_dim, num_classes, device):
        self.f0 = f0.to(device)
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.device = device
        self.learners = []
        self.alphas = []

    def add_learner(self, params, alpha):
        m = WeakLearner(self.input_dim, self.num_classes).to(self.device)
        sd = m.state_dict()
        new_sd = {k: (torch.tensor(v) if isinstance(v, np.ndarray) else v).to(self.device)
                  for k, v in zip(sd.keys(), params)}
        m.load_state_dict(new_sd)
        m.eval()
        self.learners.append(m)
        self.alphas.append(alpha)

    @torch.no_grad()
    def predict(self, x):
        out = self.f0.unsqueeze(0).expand(x.size(0), -1).clone()
        if self.learners and self.alphas:
            total = sum(self.alphas)
            if total > EPSILON:
                acc = torch.zeros_like(out)
                for m, a in zip(self.learners, self.alphas):
                    acc += (a / total) * m(x)
                out = out + acc
        return out

    def info(self):
        return {"num_learners": len(self.learners)}

## 6. Flower client

In [8]:
def make_client_fn(train_dataset, partitions, ensemble, input_dim, num_classes):
    def client_fn(cid):
        cid_int = int(cid)
        loader = DataLoader(Subset(train_dataset, partitions[cid_int]),
                            batch_size=BATCH_SIZE, shuffle=True)

        class BoostFLClient(fl.client.NumPyClient):
            def __init__(self):
                self.net = WeakLearner(input_dim, num_classes).to(DEVICE)

            def get_parameters(self, config=None):
                return get_ndarrays(self.net)

            def fit(self, parameters, config):
                set_ndarrays(self.net, parameters)
                self.net.train()
                optimizer = optim.Adam(self.net.parameters(), lr=LEARNING_RATE,
                                       weight_decay=1e-4)
                residual_loss_fn = ResidualLoss()
                total_loss, n_batches = 0.0, 0
                for _ in range(EPOCHS):
                    for xb, yb in loader:
                        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                        optimizer.zero_grad()
                        logits = self.net(xb)
                        with torch.no_grad():
                            ens_probs = torch.softmax(ensemble.predict(xb), dim=1)
                            y_oh = torch.zeros(yb.size(0), num_classes, device=DEVICE)
                            y_oh.scatter_(1, yb.unsqueeze(1), 1)
                            residuals = y_oh - ens_probs
                        loss = residual_loss_fn(logits, residuals)
                        loss.backward()
                        optimizer.step()
                        total_loss += loss.item()
                        n_batches += 1
                avg_loss = min(max(total_loss / max(n_batches, 1), EPSILON), 10.0)
                alpha = 1.0 / (1.0 + avg_loss)
                alpha = max(MIN_ALPHA, min(MAX_ALPHA, alpha))
                return (get_ndarrays(self.net),
                        len(partitions[cid_int]),
                        {"alpha": float(alpha)})

            def evaluate(self, parameters, config):
                return 0.0, len(partitions[cid_int]), {}

        return BoostFLClient().to_client()

    return client_fn

## 7. Boosting strategy (server-side aggregation + ensemble)

In [9]:
eval_rounds, eval_acc, eval_prec, eval_rec, eval_f1 = [], [], [], [], []


def reset_histories():
    eval_rounds.clear(); eval_acc.clear(); eval_prec.clear()
    eval_rec.clear(); eval_f1.clear()


@torch.no_grad()
def evaluate_ensemble(ensemble, test_dataset, average="macro"):
    loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = ensemble.predict(xb)
        y_pred.extend(torch.argmax(logits, 1).cpu().numpy())
        y_true.extend(yb.numpy())
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
    }


class BoostingStrategy(fl.server.strategy.FedAvg):
    """Boosting-weighted aggregation of client parameter differences,
    growing a global ensemble evaluated on the test set each round."""
    def __init__(self, ensemble, test_dataset, input_dim, num_classes,
                 average="macro", **kwargs):
        super().__init__(**kwargs)
        self.ensemble = ensemble
        self.test_dataset = test_dataset
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.average = average
        self.current = None

    def initialize_parameters(self, client_manager):
        init_model = WeakLearner(self.input_dim, self.num_classes).to(DEVICE)
        params = get_ndarrays(init_model)
        self.current = fl.common.ndarrays_to_parameters(params)
        return self.current

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return self.current, {}
        global_params = fl.common.parameters_to_ndarrays(self.current)
        updates = [np.zeros_like(p) for p in global_params]
        weights = []
        client_params = []
        for _, fitres in results:
            params = fl.common.parameters_to_ndarrays(fitres.parameters)
            if len(params) != len(global_params):
                continue
            a = float(fitres.metrics.get("alpha", 1.0))
            if math.isnan(a) or math.isinf(a) or a <= 0:
                a = 1.0
            a = max(MIN_ALPHA, min(MAX_ALPHA, a))
            weights.append(a)
            client_params.append(params)
        wsum = sum(weights)
        for params, w in zip(client_params, weights):
            for i in range(len(updates)):
                updates[i] += w * (params[i] - global_params[i])
        if wsum > EPSILON:
            updates = [(u / wsum) * SERVER_LR for u in updates]
        else:
            updates = [np.zeros_like(p) for p in global_params]
        new_global = [gp + updates[i] for i, gp in enumerate(global_params)]
        self.current = fl.common.ndarrays_to_parameters(new_global)

        mean_alpha = float(np.mean(weights)) if weights else 1.0
        self.ensemble.add_learner(new_global, mean_alpha)

        m = evaluate_ensemble(self.ensemble, self.test_dataset, average=self.average)
        eval_rounds.append(server_round)
        eval_acc.append(m["accuracy"])
        eval_prec.append(m["precision"])
        eval_rec.append(m["recall"])
        eval_f1.append(m["f1"])
        print(f"[round {server_round:02d}] acc={m['accuracy']:.4f} f1={m['f1']:.4f} "
              f"mean_alpha={mean_alpha:.3f} learners={self.ensemble.info()['num_learners']}")
        return self.current, {"accuracy": m["accuracy"], "f1": m["f1"],
                              "mean_alpha": mean_alpha}

    def aggregate_evaluate(self, server_round, results, failures):
        return 0.0, {}

## 8. Federated driver (Flower simulation)

In [10]:
def run_boostfl(train_dataset, test_dataset, input_dim, num_classes,
                num_rounds=NUM_ROUNDS, average="macro"):
    set_seed(SEED)
    reset_histories()

    partitions = partition_iid(train_dataset, NUM_PARTITIONS, num_classes)
    f0 = torch.randn(num_classes, device=DEVICE) * 0.01
    ensemble = Ensemble(f0, input_dim, num_classes, DEVICE)

    client_fn = make_client_fn(train_dataset, partitions, ensemble,
                               input_dim, num_classes)
    strategy = BoostingStrategy(
        ensemble=ensemble, test_dataset=test_dataset,
        input_dim=input_dim, num_classes=num_classes, average=average,
        fraction_fit=1.0, min_fit_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )

    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=num_rounds),
        strategy=strategy,
        client_resources={"num_cpus": CPUS_PER_CLIENT, "num_gpus": GPU_PER_CLIENT},
    )

    final = evaluate_ensemble(ensemble, test_dataset, average=average)
    history = {"round": list(eval_rounds), "accuracy": list(eval_acc),
               "precision": list(eval_prec), "recall": list(eval_rec),
               "f1": list(eval_f1)}
    return history, final

## 9. Multiclass run

In [11]:
train_mc, test_mc, class_names_mc, num_classes_mc, input_dim = load_unsw(
    DATA_PATH, target_column="attack_cat")
print(f"Classes: {class_names_mc}")
print(f"Features: {input_dim} | Train: {len(train_mc)} | Test: {len(test_mc)}")

hist_mc, final_mc = run_boostfl(train_mc, test_mc, input_dim, num_classes_mc)
print("\nBoostFL-IoT (multiclass):")
for k in ["accuracy", "precision", "recall", "f1"]:
    print(f"  {k}: {final_mc[k]:.4f}")

Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Features: 47 | Train: 1467428 | Test: 366858


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=15, no round_timeout
2026-09-18 13:58:41,173	INFO worker.py:1771 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'node:172.24.90.50': 1.0, 'accelerator_type:G': 1.0, 'node:__internal_head__': 1.0, 'CPU': 20.0, 'memory': 11536945152.0, 'object_store_memory': 5768472576.0, 'GPU': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 2, 'num_gpus': 0.5}
INFO :      F

[round 01] acc=0.9557 f1=0.0977 mean_alpha=0.998 learners=1


(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001032) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 2x across cluster]
(ClientAppActor pid=1001032)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=1001032)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)    

[round 02] acc=0.9691 f1=0.2382 mean_alpha=0.998 learners=2


...

[round 14] acc=0.9756 f1=0.2932 mean_alpha=0.998 learners=14


(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001032) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 2x across cluster]
(ClientAppActor pid=1001032)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=1001032)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)    

[round 15] acc=0.9756 f1=0.2935 mean_alpha=0.998 learners=15


(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001031) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 2x across cluster]
(ClientAppActor pid=1001031)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=1001031)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)         
(ClientAppActor pid=1001032) 
(ClientAppActor pid=1001032)         
(ClientAppActor pid=1001031) 
(ClientAppActor pid=1001031)    


BoostFL-IoT (multiclass):
  accuracy: 0.9756
  precision: 0.3006
  recall: 0.2989
  f1: 0.2935


## 10. Binary run

In [12]:
train_bin, test_bin, class_names_bin, num_classes_bin = to_binary(
    train_mc, test_mc, class_names_mc)

hist_bin, final_bin = run_boostfl(
    train_bin, test_bin, input_dim, num_classes_bin, average="binary")
print("\nBoostFL-IoT (binary):")
for k in ["accuracy", "precision", "recall", "f1"]:
    print(f"  {k}: {final_bin[k]:.4f}")

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=15, no round_timeout
2026-09-18 15:44:09,082	INFO worker.py:1771 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'node:172.24.90.50': 1.0, 'accelerator_type:G': 1.0, 'node:__internal_head__': 1.0, 'CPU': 20.0, 'object_store_memory': 7013083545.0, 'memory': 14026167092.0, 'GPU': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 2, 'num_gpus': 0.5}
INFO :      F

[round 01] acc=0.9557 f1=0.0048 mean_alpha=0.996 learners=1


(ClientAppActor pid=1021320) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)             This is a deprecated feature. It will be removed
(ClientAppActor pid=1021320)             entirely in future versions of Flower.
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppAct

[round 02] acc=0.9908 f1=0.8929 mean_alpha=0.997 learners=2


(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021320) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 2x across cluster]
(ClientAppActor pid=1021320)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=1021320)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)    

...

[round 14] acc=0.9913 f1=0.9027 mean_alpha=0.997 learners=14


(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021319) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 2x across cluster]
(ClientAppActor pid=1021319)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=1021319)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)    

[round 15] acc=0.9913 f1=0.9028 mean_alpha=0.997 learners=15


(ClientAppActor pid=1021319) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)             This is a deprecated feature. It will be removed
(ClientAppActor pid=1021319)             entirely in future versions of Flower.
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppActor pid=1021319) 
(ClientAppActor pid=1021319)         
(ClientAppActor pid=1021320) 
(ClientAppActor pid=1021320)         
(ClientAppAct


BoostFL-IoT (binary):
  accuracy: 0.9913
  precision: 0.8968
  recall: 0.9089
  f1: 0.9028
